# Incremental Ingestion

In [0]:
"""
Using Databricks Auto-Loader & Structured Streaming  
"Secara otomatis membaca hanya file baru yang masuk ke folder, lalu menyimpannya ke Delta Lake tanpa membaca ulang file yang sudah pernah diproses."

Di Checkpoint ada 2 save/folder:
- 1. SchemaFolder --> untuk menyimpan SCHEMA dari file-file yang sudah diproses (_schema)
- 2. RocksDB      --> untuk menyimpan riwayat dari file-file sap saja yang sudah diproses (sources/rocksdb)
"""

---

In [0]:
# Membuat kode menjadi dynamics tidak static..
# identifikasi 'roadblocknya' dulu.. kemudian jadikan ia menjadi dinamis tidak lagi statis
dbutils.widgets.text("source_data", "")

In [0]:
source_data = dbutils.widgets.get("source_data")
source_data

In [0]:
df = (
    spark.readStream
    .format("cloudFiles") # Format Auto-Loader
    .option("cloudFiles.format", "csv") # Memberitahu Auto Loader: Semua file yang akan saya baca adalah CSV.
    .option("cloudFiles.schemaLocation", f"/Volumes/flyproject/bronze/bronze_volume/{source_data}/checkpoint") # Memberitahu Auto Loader untuk menyimpan metadata schema di ...
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(f"/Volumes/flyproject/source/raw_volume/raw_data/{source_data}/") # Ini adalah folder yang dipantau.
)

# Jawabannya menarik karena Anda tidak pernah melihat baris kode yang secara eksplisit membaca checkpoint.
# Jadi checkpoint OTOMATIS dibaca saat menread
# membuat folder bookings/checkpoint

In [0]:
# Menulis hasil streaming
query = (
    df.writeStream
        .format("delta")
        .outputMode("append")
        .trigger(once=True)
        .option("checkpointLocation", f"/Volumes/flyproject/bronze/bronze_volume/{source_data}/checkpoint") # Menyuruh auto-loader untuk mengupdate checkpoint pada lokasi .., Ini adalah bagian paling penting dari streaming.
        .option("path", f"/Volumes/flyproject/bronze/bronze_volume/{source_data}/data") # Ini adalah lokasi penyimpanan hasil. 
        .start()
)
query.awaitTermination()

# biasanya bronze tidak dibuatkan table
# membuat folder bookings/data

In [0]:
# %sql
# SELECT *
# FROM delta.`/Volumes/flyproject/bronze/bronze_volume/bookings/data/`